<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/urcdp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime
import sqlite3

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
# %%
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

BASE_URL = "https://www.gub.uy/unidad-reguladora-control-datos-personales/comunicacion/noticias"
print("✅ Scraper da URCDP (Uruguay) pronto!")

✅ Scraper da URCDP (Uruguay) pronto!


In [3]:
DATABASE_NAME = "internet_governance_news.db"

def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela 'articles' prontos!")

create_database()


✅ Banco e tabela 'articles' prontos!


In [4]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [5]:
# %%
def load_articles_from_db():
    """Carrega os artigos salvos no banco SQLite para um DataFrame."""
    try:
        conn = sqlite3.connect(DATABASE_NAME)
        df = pd.read_sql("""
            SELECT id, title, date, author, url, source
            FROM articles
            ORDER BY date DESC
        """, conn)
        conn.close()
        return df
    except Exception as e:
        print(f"⚠️ Erro ao acessar o banco: {e}")
        return pd.DataFrame()

# Recarrega o dataframe para refletir o que foi coletado pelo scraper
df_db = load_articles_from_db()

if not df_db.empty:
    print(f"📦 Total no banco: {len(df_db)} registros")
    display(df_db.head(10))
else:
    print("⚠️ O banco de dados ainda está vazio. Certifique-se de executar a célula do scraper (a7b8c9d0) primeiro.")

📦 Total no banco: 10 registros


,id,title,date,author,url,source
0,10,Curso gratuito sobre protección de datos perso...,31/07/2025,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
1,1,Cuidá tus datos: recomendaciones para proteger...,28/01/2026,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
2,4,Desafíos de la protección de datos personales ...,25/11/2025,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
3,5,Liceos ganadores del concurso Competencia Digi...,22/11/2025,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
4,6,La URCDP apoyó la 76ª Reunión del Grupo de Ber...,21/11/2025,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
5,7,La URCDP participó en reuniones internacionale...,14/10/2025,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
6,2,Mecanismos de coordinación y respuesta para la...,09/12/2025,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
7,3,La URCDP participó en la XVI Asamblea General ...,05/12/2025,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
8,8,Charlas de Café: desafíos de la protección de ...,01/10/2025,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
9,9,Curso virtual sobre el derecho a la protección...,01/09/2025,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay


In [6]:
# %%
def montar_url(pagina):
    """Monta a URL paginada da URCDP (page=0 é a primeira)."""
    if pagina == 0:
        return BASE_URL
    return f"{BASE_URL}?page={pagina}"

def extrair_paragrafos(url):
    """Acessa a página da notícia e extrai os parágrafos principais."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        ps = soup.find_all("p")
        textos = [
            p.get_text(strip=True)
            for p in ps
            if len(p.get_text(strip=True).split()) > 10
        ]
        return textos[:5] if textos else ["NA"]
    except:
        return ["NA"]

In [7]:
# %%
noticias = []
TOTAL_PAGES = 25  # margem para cobrir todas as páginas (0-indexed)

for pagina in range(0, TOTAL_PAGES):
    url = montar_url(pagina)
    print(f"📄 Coletando página {pagina + 1}/{TOTAL_PAGES}: {url}")

    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            print(f"⚠️ Erro HTTP {r.status_code} — parando.")
            break
    except Exception as e:
        print(f"⚠️ Erro de conexão: {e}")
        break

    soup = BeautifulSoup(r.text, "html.parser")

    # ===== Seletores para gub.uy (Drupal)
    # As notícias aparecem como <h3><a href="...">título</a></h3> na listagem
    titulo_tags = soup.select("h3 a[href*='/comunicacion/noticias/']")

    if not titulo_tags:
        print("🏁 Sem mais resultados — fim da coleta.")
        break

    print(f"   {len(titulo_tags)} notícias encontradas")

    for titulo_tag in titulo_tags:
        titulo = titulo_tag.get_text(strip=True)
        link = titulo_tag.get("href", "")
        if link.startswith("/"):
            link = f"https://www.gub.uy{link}"

        # data — procurar no elemento pai (li) o texto com formato DD/MM/YYYY
        parent_li = titulo_tag.find_parent("li")
        data_raw = "NA"
        if parent_li:
            # procura time tag primeiro
            time_tag = parent_li.find("time")
            if time_tag:
                data_raw = time_tag.get("datetime", time_tag.get_text(strip=True))
            else:
                # fallback: regex para DD/MM/YYYY no texto do li
                texto_li = parent_li.get_text()
                match = re.search(r'\d{2}/\d{2}/\d{4}', texto_li)
                if match:
                    data_raw = match.group()

        # extrai parágrafos da notícia
        paragrafos = extrair_paragrafos(link)

        # acumula
        noticias.append({
            "titulo": titulo,
            "data": data_raw,
            "link": link,
            "paragrafos": " || ".join(paragrafos),
            "fonte": "URCDP Uruguay"
        })

        # grava no banco
        insert_article(
            title=titulo,
            date=data_raw,
            author="URCDP",
            url=link,
            source="URCDP Uruguay"
        )

    time.sleep(1)

print(f"\n✅ Total coletado: {len(noticias)} notícias")

df_urcdp = pd.DataFrame(noticias)
display(df_urcdp.head())

📄 Coletando página 1/25: https://www.gub.uy/unidad-reguladora-control-datos-personales/comunicacion/noticias
   10 notícias encontradas
📄 Coletando página 2/25: https://www.gub.uy/unidad-reguladora-control-datos-personales/comunicacion/noticias?page=1
   10 notícias encontradas
📄 Coletando página 3/25: https://www.gub.uy/unidad-reguladora-control-datos-personales/comunicacion/noticias?page=2
   10 notícias encontradas
📄 Coletando página 4/25: https://www.gub.uy/unidad-reguladora-control-datos-personales/comunicacion/noticias?page=3
   10 notícias encontradas
📄 Coletando página 5/25: https://www.gub.uy/unidad-reguladora-control-datos-personales/comunicacion/noticias?page=4
   10 notícias encontradas
📄 Coletando página 6/25: https://www.gub.uy/unidad-reguladora-control-datos-personales/comunicacion/noticias?page=5
   10 notícias encontradas
📄 Coletando página 7/25: https://www.gub.uy/unidad-reguladora-control-datos-personales/comunicacion/noticias?page=6
   10 notícias encontradas
📄 Cole

,titulo,data,link,paragrafos,fonte
0,Cuidá tus datos: recomendaciones para proteger...,28/01/2026,https://www.gub.uy/unidad-reguladora-control-d...,"En Uruguay, laLey N° 18.331, de 11 de agosto d...",URCDP Uruguay
1,Mecanismos de coordinación y respuesta para la...,09/12/2025,https://www.gub.uy/unidad-reguladora-control-d...,"Durante el encuentro, se abordaron losprocedim...",URCDP Uruguay
2,La URCDP participó en la XVI Asamblea General ...,05/12/2025,https://www.gub.uy/unidad-reguladora-control-d...,Durante el debate del Consejo sobre protección...,URCDP Uruguay
3,Desafíos de la protección de datos personales ...,25/11/2025,https://www.gub.uy/unidad-reguladora-control-d...,La apertura del evento estuvo a cargo de Feder...,URCDP Uruguay
4,Liceos ganadores del concurso Competencia Digi...,22/11/2025,https://www.gub.uy/unidad-reguladora-control-d...,Esta edición fue la primera experiencia dirigi...,URCDP Uruguay


In [8]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT * FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head(20))

📦 Total no banco: 214 registros


,id,title,date,author,url,source
0,113,Así pasó la 5ta Semana Nacional de Protección ...,31/08/2020,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
1,10,Curso gratuito sobre protección de datos perso...,31/07/2025,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
2,179,Se realizó una nueva instancia del ciclo Charl...,31/07/2018,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
3,29,Conferencia internacional sobre privacidad y p...,31/05/2024,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
4,55,Nuevo canal disponible para realizar solicitud...,31/01/2023,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
5,16,Saludo de fin de año 2024,30/12/2024,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
6,75,Sigamos trabajando en conjunto ¡feliz 2022!,30/12/2021,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
7,39,Capacitación sobre transferencias internaciona...,30/11/2023,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
8,108,Avanzamos en el plan de capacitación anual de ...,30/11/2020,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay
9,192,URCDP realizó actividad en la Intendencia de T...,30/04/2018,URCDP,https://www.gub.uy/unidad-reguladora-control-d...,URCDP Uruguay


In [9]:
keywords = ['digital', 'internet', 'IA', 'tecnología', 'datos', 'privacidad',
            'protección', 'ciberseguridad', 'inteligencia artificial']
pattern = r'|'.join(keywords)

df_filt = df_urcdp[
    df_urcdp['titulo'].str.contains(pattern, case=False, na=False, regex=True) |
    df_urcdp['paragrafos'].str.contains(pattern, case=False, na=False, regex=True)
].copy()

print(f"{len(df_filt)} notícias filtradas (de {len(df_urcdp)})")
display(df_filt.head())

212 notícias filtradas (de 214)


,titulo,data,link,paragrafos,fonte
0,Cuidá tus datos: recomendaciones para proteger...,28/01/2026,https://www.gub.uy/unidad-reguladora-control-d...,"En Uruguay, laLey N° 18.331, de 11 de agosto d...",URCDP Uruguay
1,Mecanismos de coordinación y respuesta para la...,09/12/2025,https://www.gub.uy/unidad-reguladora-control-d...,"Durante el encuentro, se abordaron losprocedim...",URCDP Uruguay
2,La URCDP participó en la XVI Asamblea General ...,05/12/2025,https://www.gub.uy/unidad-reguladora-control-d...,Durante el debate del Consejo sobre protección...,URCDP Uruguay
3,Desafíos de la protección de datos personales ...,25/11/2025,https://www.gub.uy/unidad-reguladora-control-d...,La apertura del evento estuvo a cargo de Feder...,URCDP Uruguay
4,Liceos ganadores del concurso Competencia Digi...,22/11/2025,https://www.gub.uy/unidad-reguladora-control-d...,Esta edición fue la primera experiencia dirigi...,URCDP Uruguay


In [13]:
import plotly.io as pio
pio.renderers.default = 'colab'

def plot_charts(df):
    if df.empty:
        print("❌ Sem dados para gráficos")
        return

    # ------------------------------
    # Top 15 - Notícias
    # ------------------------------
    # Garantindo que pegamos as mais recentes ou relevantes
    top15 = df.head(15).copy()

    fig1 = px.bar(
        top15,
        x='date',
        y='title',
        orientation='h',
        title='Top 15 Notícias – Recentes',
        labels={'date': 'Data', 'title': 'Título'}
    )
    fig1.update_layout(height=600, yaxis={'categoryorder':'total ascending'})
    fig1.show()

    # ------------------------------
    # Pizza por Fonte
    # ------------------------------
    source_count = df["source"].value_counts().reset_index()
    source_count.columns = ["source", "count"]

    fig2 = px.pie(
        source_count,
        names="source",
        values="count",
        title="Distribuição por Fonte"
    )
    fig2.show()

    # ------------------------------
    # Treemap de Palavras Comuns
    # ------------------------------
    text = ' '.join(df['title'].astype(str)).lower()
    words = re.findall(r'\b\w{5,}\b', text) # palavras com mais de 5 letras

    wc = (
        pd.Series(words)
        .value_counts()
        .head(20)
        .reset_index()
    )
    wc.columns = ['palavra', 'freq']

    fig3 = px.treemap(
        wc,
        path=['palavra'],
        values='freq',
        title='Palavras Mais Frequentes nos Títulos'
    )
    fig3.show()

In [14]:
# Garante que os dados do banco sejam carregados e os gráficos gerados
df_db = load_articles_from_db()
if not df_db.empty:
    plot_charts(df_db)
else:
    print("☑ No h dados no banco para gerar os grficos.")

In [15]:
def plot_extra_charts(df):
    if df.empty:
        print("❌ Sem dados para gráficos extras")
        return

    # Converter data para datetime para análise temporal
    df['date_dt'] = pd.to_datetime(df['date'], format='%d/%m/%Y', errors='coerce')
    df_time = df.dropna(subset=['date_dt']).copy()

    # 1. Evolução Temporal (Notícias por Mês)
    df_time['mes_ano'] = df_time['date_dt'].dt.to_period('M').astype(str)
    timeline = df_time.groupby('mes_ano').size().reset_index(name='quantidade')

    fig4 = px.line(
        timeline,
        x='mes_ano',
        y='quantidade',
        title='Frequência de Publicações ao Longo do Tempo',
        markers=True
    )
    fig4.update_xaxes(tickangle=45)
    fig4.show()

    # 2. Top Autores/Entidades
    author_count = df['author'].value_counts().head(10).reset_index()
    author_count.columns = ['autor', 'contagem']

    fig5 = px.bar(
        author_count,
        x='contagem',
        y='autor',
        orientation='h',
        title='Top 10 Autores/Entidades',
        color='contagem',
        color_continuous_scale='Viridis'
    )
    fig5.show()

plot_extra_charts(df_db)

In [16]:
# Criando um dicionário para armazenar as contagens
keyword_counts = {}

# Concatenando título e parágrafos para uma busca completa em cada linha
df_filt['texto_completo'] = df_filt['titulo'].astype(str) + " " + df_filt['paragrafos'].astype(str)

for kw in keywords:
    # Conta em quantas notícias a palavra-chave aparece (case-insensitive)
    count = df_filt['texto_completo'].str.contains(kw, case=False, na=False).sum()
    keyword_counts[kw] = count

# Convertendo para DataFrame para melhor visualização
df_keywords = pd.DataFrame(list(keyword_counts.items()), columns=['Palavra-Chave', 'Frequência'])
df_keywords = df_keywords.sort_values(by='Frequência', ascending=False)

print("📊 Frequência de Palavras-Chave nas Notícias Filtradas:")
display(df_keywords)

# Visualização rápida
fig_kw = px.bar(df_keywords, x='Palavra-Chave', y='Frequência', title='Prevalência de Temas (Palavras-Chave)', color='Frequência')
fig_kw.show()

📊 Frequência de Palavras-Chave nas Notícias Filtradas:


,Palavra-Chave,Frequência
4,datos,205
2,IA,203
6,protección,193
0,digital,56
5,privacidad,53
3,tecnología,37
8,inteligencia artificial,28
1,internet,15
7,ciberseguridad,5


### Análise de Correlação entre Palavras-Chave
Nesta seção, calculamos como a presença de uma palavra-chave se relaciona com a presença de outra no mesmo texto.

In [17]:
import numpy as np

# Criar uma matriz binária (One-Hot Encoding manual) para as palavras-chave
# Cada linha é uma notícia, cada coluna é uma keyword (1 se presente, 0 se não)
correlation_data = pd.DataFrame()

for kw in keywords:
    correlation_data[kw] = df_filt['texto_completo'].str.contains(kw, case=False, na=False).astype(int)

# Calcular a matriz de correlação
corr_matrix = correlation_data.corr()

# Visualizar com um Heatmap do Plotly
fig_corr = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect="auto",
    color_continuous_scale='RdBu_r',
    title="Matriz de Correlação entre Palavras-Chave (Presença Simultânea)",
    labels=dict(color="Correlação")
)

fig_corr.show()

# Exibir as correlações mais fortes (excluindo a diagonal)
sorted_corr = corr_matrix.unstack().sort_values(ascending=False)
sorted_corr = sorted_corr[sorted_corr < 1.0].drop_duplicates()

print("Top 5 associações mais fortes entre temas:")
display(sorted_corr.head(5))

Top 5 associações mais fortes entre temas:


,,0
datos,protección,0.588944
tecnología,digital,0.288259
internet,tecnología,0.260848
inteligencia artificial,ciberseguridad,0.214795
digital,ciberseguridad,0.188896
